# Map annotations to the full 6k comb object

## Rationale

Maintaining a full 6K-gene `comb` object ensures access to the complete CosMx panel rather than restricting downstream analyses to the top 2,000 highly variable genes. This is important for analyses that benefit from broader gene coverage, including spatial niche identification with NOVAE, pseudobulk differential expression with DESeq2, and ligand–receptor interaction analysis with CellChat.

## Code

Imports

In [1]:
# %% Libraries
import os
import scanpy as sc
import numpy as np
from pathlib import Path
import rapids_singlecell as rsc
import pandas as pd
from scipy.sparse import issparse

# %% Setting Paths
MAIN_DIR_NAME = "proseg_data"
MAIN_DIR = next(p for p in Path.cwd().parents if (p / MAIN_DIR_NAME).exists()) / MAIN_DIR_NAME
os.chdir(MAIN_DIR)

# %% Setting Seed
SEED_VALUE = 42
# set NumPy RNG for consistency
np.random.seed(SEED_VALUE)

# %% object versions
COMB_V= 'cellcharter'
COMB_PATH = MAIN_DIR / 'data' / 'comb' / 'h5ad' / f'comb-{COMB_V}.h5ad'

# unintegrated is the last 6k version after QC
UNINTEGRATED_COMB_PATH = MAIN_DIR / 'data' / 'comb' / 'h5ad' / 'comb.h5ad'

# new object with 6k genes and annotations
FULL_COMB_V = 'annotated'
FULL_COMB_PATH = MAIN_DIR / 'data' / 'comb-full' / 'h5ad' / f'comb-full-{FULL_COMB_V}.h5ad'

/mnt/data/project0062/.conda/envs/rsc_25.12/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# load comb obj
comb = sc.read_h5ad(COMB_PATH)

In [13]:
# load airway epi obj
comb_full = sc.read_h5ad(UNINTEGRATED_COMB_PATH)

Subset to keep only cells in comb

In [14]:
cells_keep = comb.obs_names

In [15]:
comb_full = comb_full[cells_keep, :].copy()

In [16]:
comb_full

AnnData object with n_obs × n_vars = 1187341 × 6175
    obs: 'cell', 'original_cell_id', 'centroid_x', 'centroid_y', 'centroid_z', 'component', 'volume', 'surface_area', 'scale', 'slide_ID', 'cell_id', 'sample_name'
    obsm: 'spatial'

Create coords in `comb_full`

In [19]:
# %% define X_spatial
comb_full.obsm["X_spatial"] = np.array(
    comb_full.obs[['centroid_x', 'centroid_y']]
)

# %% Normalization and log1p transformation
comb_full.layers['counts'] = comb_full.X.copy() # saving the raw counts in a layer
sc.pp.normalize_total(comb_full, target_sum=1e4)
sc.pp.log1p(comb_full)

# Save normalized + log1p expression
comb_full.layers["normalized"] = comb_full.X.copy()

# Save normalized + log1p expression as .raw
comb_full.raw = comb_full.copy() # this is just a snapshot of normalized+1logp counts. Used by default for plots.

# show comb_full 
comb_full

AnnData object with n_obs × n_vars = 1187341 × 6175
    obs: 'cell', 'original_cell_id', 'centroid_x', 'centroid_y', 'centroid_z', 'component', 'volume', 'surface_area', 'scale', 'slide_ID', 'cell_id', 'sample_name'
    uns: 'log1p'
    obsm: 'spatial', 'X_spatial'
    layers: 'counts', 'normalized'

Just need to replace the metadata

In [20]:
ann_meta = comb.obs
# replace in comb_full
comb_full.obs = ann_meta

In [21]:
print(comb_full.obs)

                  cell original_cell_id   centroid_x  centroid_y  centroid_z  \
cell_id                                                                        
SLIDE04_0            0     c_4_148_1200    389.99450   2561.8610    0.578297   
SLIDE04_2            2     c_4_160_1452    937.05630   2050.9023    0.365066   
SLIDE04_3            3     c_4_160_1451   1022.33000   2049.8896    0.447500   
SLIDE04_4            4     c_4_160_1324    726.74194   2110.9038    0.526499   
SLIDE04_5            5     c_4_160_1346    750.20110   2100.1968    0.524456   
...                ...              ...          ...         ...         ...   
SLIDE02_426982  426982      c_2_113_381  11781.21800  13423.0800    0.815171   
SLIDE02_426983  426983      c_2_113_366  11778.81100  13431.9150    0.765805   
SLIDE02_426985  426985      c_2_113_317  11782.32000  13455.4850    0.865000   
SLIDE02_426986  426986      c_2_113_310  11777.96300  13463.0020    0.794118   
SLIDE02_426987  426987      c_2_113_291 

Save the annotated object

In [22]:
# cretate the directory if it does not exist
FULL_COMB_PATH.parent.mkdir(parents=True, exist_ok=True)

In [23]:
comb_full

AnnData object with n_obs × n_vars = 1187341 × 6175
    obs: 'cell', 'original_cell_id', 'centroid_x', 'centroid_y', 'centroid_z', 'component', 'volume', 'surface_area', 'scale', 'slide_ID', 'sample_name', 'n_genes', 'slide_name', 'condition', 'donor', 'run', 'modulator', 'age', 'mutation', 'sex', '_indices', '_scvi_batch', '_scvi_ind_x', '_scvi_labels', 'leiden_resolvi_0.1', 'leiden_resolvi_0.2', 'leiden_resolvi_0.3', 'leiden_resolvi_0.4', 'leiden_resolvi_0.5', 'leiden_resolvi_0.6', 'leiden_resolvi_0.7', 'leiden_resolvi_0.8', 'leiden_resolvi_0.9', 'leiden_resolvi_1.0', 'leiden_resolvi_1.1', 'leiden_resolvi_1.2', 'leiden_resolvi_1.3', 'leiden_resolvi_1.4', 'leiden_resolvi_1.5', 'ct_resolvi', 'ann_lvl_3', 'airway-epithelium_leiden_n10_0.4', 'ann_lvl_3_refined', 'ambiguous_leiden_n10_0.2', 'plasma_leiden_n10_0.3', 'T_leiden_n10_0.3', 'B_leiden_n10_0.2', 'endothelial_leiden_n10_0.2', 'ann_lvl_2', 'ann_lvl_1', 'niches_k_1', 'niches_k_2', 'niches_k_3', 'niches_k_4', 'niches_k_5', 'niches_k_

In [24]:
comb_full.write_h5ad(FULL_COMB_PATH)